# 02 - Mixed-Frequency Dynamic Factor Model (DFM-MF)

## Overview

The **DFM with mixed frequencies** (DFM-MF) handles series observed at
different frequencies — e.g., monthly indicators alongside quarterly GDP.
This is a natural application of the Kalman filter, which propagates
information optimally even when some observations are missing.

### Key Idea

- Monthly series ($y^m_t$): observed every period
- Quarterly series ($y^q_t$): observed only in quarter-end months (March,
  June, September, December); coded as `NaN` in other months
- The Kalman filter processes all available information at each time step,
  skipping missing entries in the observation equation

### Nowcasting

By running the filter up to the current month, we can produce a **nowcast**
of quarterly GDP using the monthly indicators available so far in the
quarter. As more monthly data arrives, the nowcast improves — this is
the "information gain" across the quarter.

### Model

$$
\begin{align}
y_t &= \Lambda f_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, R) \\
f_t &= \Phi f_{t-1} + \eta_t, \quad \eta_t \sim N(0, I)
\end{align}
$$

where $y_t$ may have `NaN` entries for unobserved series at time $t$.

### This Notebook

We use `mixed_freq_macro.csv` with 4 monthly series (industrial_production,
unemployment, cpi, pmi) and 1 quarterly series (gdp_growth). We demonstrate:
- Model estimation with missing data
- Nowcasting of GDP
- Information gain through the quarter
- Pseudo out-of-sample nowcast evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from kalmanbox import DynamicFactorModel

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

print('Imports OK')

In [ ]:
# Load mixed-frequency dataset
data_dir = Path(__file__).resolve().parent / 'data' if '__file__' in dir() else Path('data')
df = pd.read_csv(data_dir / 'mixed_freq_macro.csv', parse_dates=['date'])
df = df.set_index('date')

series_names = df.columns.tolist()
print(f'Shape: {df.shape}')
print(f'Period: {df.index[0].strftime("%Y-%m")} to {df.index[-1].strftime("%Y-%m")}')
print(f'Series: {series_names}')
print(f'\nMissing values per series:')
print(df.isnull().sum())
print(f'\nFirst 15 rows (note NaN pattern in GDP):')
df.head(15)

## Visualizing the Missing Data Pattern

GDP is observed only in quarter-end months (every 3rd month). The monthly
indicators are fully observed.

In [ ]:
# Missing data pattern visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel 1: missing data heatmap (first 48 months = 4 years)
n_show = min(48, len(df))
missing_mat = df.iloc[:n_show].isnull().astype(int).T
axes[0].imshow(missing_mat, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')
axes[0].set_yticks(range(len(series_names)))
axes[0].set_yticklabels(series_names)
axes[0].set_xlabel('Month index')
axes[0].set_title('Missing Data Pattern (red = missing, green = observed)')
# Add legend
observed_patch = mpatches.Patch(color='#1a9641', label='Observed')
missing_patch = mpatches.Patch(color='#d7191c', label='Missing (NaN)')
axes[0].legend(handles=[observed_patch, missing_patch], loc='upper right')

# Panel 2: Time series with observed GDP points highlighted
dates = df.index
ax = axes[1]
for i, col in enumerate(series_names):
    if col == 'gdp_growth':
        observed_mask = ~df[col].isnull()
        ax.plot(dates[observed_mask], df[col][observed_mask], 'ro', markersize=5,
                label='GDP (quarterly, observed)', zorder=5)
    else:
        ax.plot(dates, df[col], alpha=0.4, linewidth=0.8, label=col)
ax.set_ylabel('Standardized Value')
ax.set_title('Mixed-Frequency Panel: Monthly Series + Quarterly GDP')
ax.legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.show()

## DFM-MF: Model Specification and Estimation

The DFM naturally handles the mixed-frequency structure. We pass the data
with NaN values — the Kalman filter adjusts the observation equation at
each time step, using only the available series.

In [ ]:
# Fit DFM with K=1 factor on mixed-frequency data (NaN in GDP)
y_mf = df.values.astype(np.float64)

model_mf = DynamicFactorModel(y_mf, k_factors=1, factor_order=1,
                               endog_names=series_names)
results_mf = model_mf.fit(compute_se=False)

print(results_mf.summary())

# Extracted factor
factor_mf = results_mf.smoothed_state[:, 0]
Lambda_mf = results_mf.ssm.Z[:, 0]

print(f'\nFactor loadings:')
for name, lam in zip(series_names, Lambda_mf):
    print(f'  {name:30s}: {lam:+.4f}')

## Nowcasting GDP

Using the fitted model, the **filtered state** at each time step gives us
the best estimate of the factor (and hence GDP) using only information
available up to that point. The GDP nowcast at time $t$ is:

$$\hat{y}^{GDP}_t = \lambda_{GDP} \cdot \hat{f}_{t|t}$$

where $\hat{f}_{t|t}$ is the Kalman-filtered factor estimate.

In [ ]:
# Nowcast GDP using filtered factor
gdp_idx = series_names.index('gdp_growth')
gdp_loading = Lambda_mf[gdp_idx]

# Filtered nowcast of GDP at every month
filtered_factor = results_mf.filtered_state[:, 0]
filtered_cov = results_mf.filtered_cov[:, 0, 0]
gdp_nowcast = gdp_loading * filtered_factor
gdp_nowcast_se = np.abs(gdp_loading) * np.sqrt(filtered_cov)

# Smoothed estimate (uses all data, for comparison)
gdp_smoothed = gdp_loading * factor_mf

# Observed GDP values (quarterly)
gdp_observed = df['gdp_growth'].copy()
gdp_obs_mask = ~gdp_observed.isnull()

fig, ax = plt.subplots(figsize=(14, 6))

# Plot nowcast (filtered)
ax.plot(dates, gdp_nowcast, 'b-', linewidth=1, alpha=0.7, label='GDP Nowcast (filtered)')
ax.fill_between(dates, gdp_nowcast - 1.96 * gdp_nowcast_se,
                gdp_nowcast + 1.96 * gdp_nowcast_se, alpha=0.15, color='blue')

# Plot smoothed
ax.plot(dates, gdp_smoothed, 'g-', linewidth=1, alpha=0.5, label='GDP Smoothed')

# Plot observed GDP
ax.plot(dates[gdp_obs_mask], gdp_observed[gdp_obs_mask], 'ro', markersize=6,
        label='GDP Observed (quarterly)', zorder=5)

ax.set_ylabel('GDP Growth (standardized)')
ax.set_title('GDP Nowcasting via Mixed-Frequency DFM')
ax.legend()
plt.tight_layout()
plt.show()

# Nowcast accuracy at observed quarters
gdp_obs_vals = gdp_observed[gdp_obs_mask].values
gdp_nc_vals = gdp_nowcast[gdp_obs_mask]
rmse = np.sqrt(np.mean((gdp_obs_vals - gdp_nc_vals)**2))
corr = np.corrcoef(gdp_obs_vals, gdp_nc_vals)[0, 1]
print(f'Nowcast performance (at observed quarters):')
print(f'  RMSE: {rmse:.4f}')
print(f'  Correlation: {corr:.4f}')

## Information Gain Through the Quarter

As each new month of data arrives within a quarter, the Kalman filter
incorporates the new monthly observations and updates the GDP nowcast.
We measure the **nowcast uncertainty** (filtered state variance) at each
month within the quarter: month 1 (first month), month 2, and month 3
(quarter-end, when GDP is observed).

In [ ]:
# Information gain: track nowcast uncertainty by month-in-quarter
months = df.index.month
quarter_month = np.array([(m - 1) % 3 + 1 for m in months])  # 1, 2, or 3

# Group filtered variance by month-in-quarter
variances_by_qm = {1: [], 2: [], 3: []}
for t in range(len(dates)):
    qm = quarter_month[t]
    variances_by_qm[qm].append(filtered_cov[t])

avg_var = {qm: np.mean(v) for qm, v in variances_by_qm.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of average variance by month-in-quarter
ax = axes[0]
qm_labels = ['Month 1\n(start of Q)', 'Month 2\n(mid Q)', 'Month 3\n(end of Q)']
bars = ax.bar(qm_labels, [avg_var[1], avg_var[2], avg_var[3]],
              color=['#e74c3c', '#f39c12', '#27ae60'], alpha=0.8)
ax.set_ylabel('Avg. Factor Variance (filtered)')
ax.set_title('Nowcast Uncertainty by Month-in-Quarter')
for bar, v in zip(bars, [avg_var[1], avg_var[2], avg_var[3]]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)

# Nowcast evolution for a specific quarter (example)
# Pick a quarter where GDP is observed
gdp_obs_dates = dates[gdp_obs_mask]
example_q_end = gdp_obs_dates[len(gdp_obs_dates) // 2]  # Pick middle quarter
example_q_idx = np.where(dates == example_q_end)[0][0]

ax = axes[1]
q_months = [example_q_idx - 2, example_q_idx - 1, example_q_idx]
q_dates_str = [dates[t].strftime('%Y-%m') for t in q_months]
q_nowcasts = [gdp_nowcast[t] for t in q_months]
q_se = [gdp_nowcast_se[t] for t in q_months]
actual_gdp = df['gdp_growth'].iloc[example_q_idx]

ax.errorbar(q_dates_str, q_nowcasts, yerr=[1.96 * s for s in q_se],
            fmt='bo-', linewidth=2, markersize=8, capsize=5, label='Nowcast +/- 95% CI')
ax.axhline(actual_gdp, color='red', linestyle='--', linewidth=2,
           label=f'Actual GDP = {actual_gdp:.3f}')
ax.set_xlabel('Month')
ax.set_ylabel('GDP Nowcast')
ax.set_title(f'Nowcast Evolution Within Quarter Ending {example_q_end.strftime("%Y-%m")}')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Information gain (variance reduction from month 1 to 3): '
      f'{(1 - avg_var[3] / avg_var[1]) * 100:.1f}%')

## Comparison with Monthly-Only Model

We compare the mixed-frequency model (which uses GDP observations when
available) with a model estimated using only the 4 monthly series
(ignoring GDP entirely). This shows the value of incorporating the
quarterly GDP signal.

In [ ]:
# Monthly-only model (excluding GDP column)
monthly_cols = [c for c in series_names if c != 'gdp_growth']
y_monthly = df[monthly_cols].values.astype(np.float64)

model_monthly = DynamicFactorModel(y_monthly, k_factors=1, factor_order=1,
                                    endog_names=monthly_cols)
results_monthly = model_monthly.fit(compute_se=False)

# Compare factors
factor_monthly = results_monthly.smoothed_state[:, 0]

fig, ax = plt.subplots(figsize=(14, 5))
# Standardize both factors for comparison
f_mf_std = (factor_mf - factor_mf.mean()) / factor_mf.std()
f_m_std = (factor_monthly - factor_monthly.mean()) / factor_monthly.std()

# Align sign
sign = np.sign(np.corrcoef(f_mf_std, f_m_std)[0, 1])
f_m_std = sign * f_m_std

ax.plot(dates, f_mf_std, 'b-', linewidth=1.5, label='Mixed-Frequency Factor')
ax.plot(dates, f_m_std, 'r--', linewidth=1.5, alpha=0.7, label='Monthly-Only Factor')
ax.set_ylabel('Standardized Factor')
ax.set_title('Factor Comparison: Mixed-Frequency vs Monthly-Only')
ax.legend()
plt.tight_layout()
plt.show()

corr_factors = np.abs(np.corrcoef(factor_mf, factor_monthly)[0, 1])
print(f'|Correlation| between MF and monthly-only factors: {corr_factors:.4f}')
print(f'\nMixed-freq model LogL: {results_mf.loglike:.2f} (includes GDP info)')
print(f'Monthly-only model LogL: {results_monthly.loglike:.2f} (no GDP)')

## Pseudo Out-of-Sample Nowcast Evaluation

We perform a recursive (expanding window) nowcast exercise:
- For each quarter from a start date onward, we estimate the model using
  data up to each month within the quarter and produce a GDP nowcast.
- We compare the nowcast to the actual GDP release.

In [ ]:
# Pseudo out-of-sample nowcast using expanding window
# We use the full-sample parameters (estimated above) and run the filter
# up to each time point (this is a "pseudo" exercise — true OOS would re-estimate)

gdp_obs_indices = np.where(gdp_obs_mask)[0]
# Use the second half of quarters for evaluation
n_quarters = len(gdp_obs_indices)
eval_start = n_quarters // 2
eval_quarters = gdp_obs_indices[eval_start:]

nowcast_results = []

for q_end_idx in eval_quarters:
    actual = df['gdp_growth'].iloc[q_end_idx]
    # Nowcast at month 1, 2, 3 of the quarter
    for offset, month_label in zip([2, 1, 0], ['M1', 'M2', 'M3']):
        t = q_end_idx - offset
        if t >= 0:
            nc = gdp_nowcast[t]
            nc_se = gdp_nowcast_se[t]
            nowcast_results.append({
                'quarter_end': dates[q_end_idx].strftime('%Y-%m'),
                'month_in_q': month_label,
                'nowcast': nc,
                'actual': actual,
                'error': nc - actual,
                'se': nc_se,
            })

nc_df = pd.DataFrame(nowcast_results)

# RMSE by month-in-quarter
print('Pseudo out-of-sample RMSE by month-in-quarter:')
for m in ['M1', 'M2', 'M3']:
    subset = nc_df[nc_df['month_in_q'] == m]
    rmse = np.sqrt(np.mean(subset['error']**2))
    corr = np.corrcoef(subset['nowcast'], subset['actual'])[0, 1]
    print(f'  {m}: RMSE={rmse:.4f}, Corr={corr:.4f}')

# Plot RMSE by month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rmse_by_m = [np.sqrt(np.mean(nc_df[nc_df['month_in_q'] == m]['error']**2))
             for m in ['M1', 'M2', 'M3']]
axes[0].bar(['Month 1', 'Month 2', 'Month 3'], rmse_by_m,
            color=['#e74c3c', '#f39c12', '#27ae60'], alpha=0.8)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Nowcast RMSE by Month-in-Quarter')

# Scatter: nowcast vs actual at M3 (best nowcast)
m3 = nc_df[nc_df['month_in_q'] == 'M3']
axes[1].scatter(m3['actual'], m3['nowcast'], alpha=0.6, edgecolors='k')
lims = [min(m3['actual'].min(), m3['nowcast'].min()) - 0.5,
        max(m3['actual'].max(), m3['nowcast'].max()) + 0.5]
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_xlabel('Actual GDP')
axes[1].set_ylabel('Nowcast GDP')
axes[1].set_title('Nowcast vs Actual (end-of-quarter)')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

## Conclusions

1. **Mixed-frequency handling**: The DFM naturally handles mixed-frequency
   data via the Kalman filter's ability to process missing observations.
   No special data transformation is needed — just pass NaN for unobserved
   entries.

2. **GDP Nowcasting**: The monthly indicators (industrial production,
   unemployment, CPI, PMI) provide useful real-time signal for estimating
   quarterly GDP before its official release.

3. **Information gain**: Nowcast uncertainty decreases as more monthly data
   becomes available within the quarter. The RMSE drops from month 1 to
   month 3, confirming that each new data release improves the estimate.

4. **Mixed-freq vs monthly-only**: Including the quarterly GDP observations
   (even though sparse) helps anchor the factor and improves the model.

5. **Practical use**: In real-time applications, the nowcast would be
   re-estimated as each new data point arrives, providing policymakers with
   an updated GDP estimate well before the official release.

### References

- Banbura, M., Giannone, D. and Reichlin, L. (2011). "Nowcasting."
  *Oxford Handbook of Economic Forecasting*.
- Mariano, R.S. and Murasawa, Y. (2003). "A New Coincident Index of
  Business Cycles Based on Monthly and Quarterly Series."
  *Journal of Applied Econometrics*.